In [6]:
import os
from typing import Dict, List, Tuple
from datetime import datetime
from langchain_openai import AzureChatOpenAI
from langchain.callbacks import get_openai_callback
from langchain.schema import HumanMessage, AIMessage
from langfuse import Langfuse
import json
from rich import print
from rich.console import Console
from rich.panel import Panel
from rich.prompt import Prompt, Confirm

import warnings
warnings.filterwarnings("ignore")

console = Console()

In [ ]:
langfuse = Langfuse(
  secret_key="",
  public_key="",
  host=""
)

azure_model = AzureChatOpenAI(
        openai_api_key="",
        openai_api_version="",
        azure_endpoint="",
        deployment_name='',
        model_name="gpt-4-32k",
        temperature=0.1,
        max_tokens=2000
    ) 


In [ ]:
from langchain.chat_models import AzureChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain
from langfuse.callback import CallbackHandler
import httpx
import uuid
import certifi
import ssl
import time
from typing import Dict, Any
import nltk
from nltk.tokenize import sent_tokenize
import numpy as np
from textblob import TextBlob
import re

# Download required NLTK data
nltk.download('punkt', quiet=True)

class ResponseEvaluator:
    @staticmethod
    def evaluate_response(question: str, response: str) -> Dict[str, float]:
        """Evaluate the LLM response using multiple metrics"""
        metrics = {}
        
        # Response time (calculated externally and passed to this method)
        metrics['response_time'] = 0  # Will be set in main()
        
        # Response length metrics
        metrics['word_count'] = len(response.split())
        metrics['char_count'] = len(response)
        
        # Sentence complexity
        sentences = sent_tokenize(response)
        metrics['avg_sentence_length'] = np.mean([len(sent.split()) for sent in sentences])
        metrics['sentence_count'] = len(sentences)
        
        # Question relevance score (basic keyword matching)
        question_keywords = set(question.lower().split())
        response_keywords = set(response.lower().split())
        metrics['keyword_overlap'] = len(question_keywords.intersection(response_keywords)) / len(question_keywords)
        
        # Sentiment analysis
        blob = TextBlob(response)
        metrics['sentiment_polarity'] = blob.sentiment.polarity
        metrics['sentiment_subjectivity'] = blob.sentiment.subjectivity
        
        # Readability (simplified Flesch-Kincaid)
        word_count = metrics['word_count']
        sentence_count = metrics['sentence_count']
        if sentence_count > 0:
            metrics['readability_score'] = 206.835 - 1.015 * (word_count / sentence_count)
        else:
            metrics['readability_score'] = 0
        
        return metrics

def create_custom_client():
    ssl_context = ssl.create_default_context()
    return httpx.Client(
        verify=ssl_context,
        timeout=30.0
    )

def create_chain():
    llm = AzureChatOpenAI(
        openai_api_key="",
        openai_api_version="",
        openai_api_base="",
        deployment_name='',
        model_name="gpt-4-32k",
        temperature=0.1,
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant that provides clear and accurate information."),
        ("human", "{question}")
    ])
    return LLMChain(llm=llm, prompt=prompt)

def get_user_feedback() -> Dict[str, Any]:
    """Get detailed feedback from the user"""
    feedback = {}
    
    # Numeric rating
    while True:
        try:
            feedback['rating'] = int(input("\nRate the response (0-5): "))
            if 0 <= feedback['rating'] <= 5:
                break
            print("Please enter a number between 0 and 5")
        except ValueError:
            print("Please enter a valid number")
    
    # Specific aspect ratings
    aspects = {
        'accuracy': "How accurate was the response? (0-5): ",
        'clarity': "How clear was the response? (0-5): ",
        'relevance': "How relevant was the response to your question? (0-5): "
    }
    
    for aspect, prompt in aspects.items():
        while True:
            try:
                score = int(input(prompt))
                if 0 <= score <= 5:
                    feedback[aspect] = score
                    break
                print("Please enter a number between 0 and 5")
            except ValueError:
                print("Please enter a valid number")
    
    # Optional comment
    feedback['comment'] = input("\nAny additional comments (optional): ")
    
    return feedback

In [ ]:
def main():
    try:
        custom_client = create_custom_client()
        langfuse_handler = CallbackHandler(
            public_key="",
            secret_key="",
            host="",
            httpx_client=custom_client
        )
        chain = create_chain()
        evaluator = ResponseEvaluator()
        
        while True:
            question = input("\nEnter your question (or 'quit' to exit): ")
            if question.lower() == 'quit':
                break
                
            try:
                trace_id = str(uuid.uuid4())
                langfuse_handler.trace_id = trace_id
                
                # Measure response time
                start_time = time.time()
                response = chain.invoke(
                    {"question": question},
                    config={"callbacks": [langfuse_handler]}
                )
                response_time = time.time() - start_time
                
                ai_response = response['text']
                print("\nAI Response:", ai_response)
                
                # Evaluate response
                metrics = evaluator.evaluate_response(question, ai_response)
                metrics['response_time'] = response_time
                
                # Get detailed user feedback
                feedback = get_user_feedback()
                
                # Log comprehensive feedback to Langfuse
                langfuse_handler.langfuse.score(
                    trace_id=trace_id,
                    name="user_feedback",
                    value=feedback['rating'],
                    comment=f"Overall Rating: {feedback['rating']}/5\n"
                            f"Accuracy: {feedback['accuracy']}/5\n"
                            f"Clarity: {feedback['clarity']}/5\n"
                            f"Relevance: {feedback['relevance']}/5\n"
                            f"User Comment: {feedback['comment']}"
                )
                
                # Log metrics to Langfuse
                langfuse_handler.langfuse.score(
                    trace_id=trace_id,
                    name="response_metrics",
                    value=metrics['keyword_overlap'],  # Using relevance score as main metric
                    comment=str(metrics)
                )
                
                print("\nFeedback and metrics recorded successfully!")
                
            except Exception as e:
                print(f"Error during interaction: {str(e)}")
                if langfuse_handler and trace_id:
                    try:
                        langfuse_handler.langfuse.score(
                            trace_id=trace_id,
                            name="error",
                            value=-1,
                            comment=str(e)
                        )
                    except Exception as error_e:
                        print(f"Failed to log error: {str(error_e)}")
                        
    except Exception as e:
        print(f"Failed to initialize: {str(e)}")
    finally:
        if 'custom_client' in locals():
            custom_client.close()

if __name__ == "__main__":
    main()

AI Response: NLP stands for Natural Language Processing. It is a branch of artificial intelligence that deals with 
the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, 
decipher, understand, and make sense of the human language in a valuable way. It involves several challenges, 
including speech recognition, natural language understanding, and natural language generation.

Feedback and metrics recorded successfully!

AI Response: Generative AI is a subset of artificial intelligence that leverages machine learning techniques to 
generate human-like content. It can create new content that it has never seen before, such as written text, images,
videos, music, or voice. It learns patterns, structures, and elements from existing data and then uses this 
knowledge to create new, original content. 

Examples of Generative AI include chatbots that can carry on a conversation, AI that can write poetry or stories, 
or AI that can create realistic images or compose music. Some of the popular models used in Generative AI include 
Generative Adversarial Networks (GANs), Variational Autoencoders (VAEs), and Transformer models like GPT-3.

Feedback and metrics recorded successfully!